# Setup

In [1]:
# transformer-lens 3.4.0 pins torchvision>=0.22,<0.23 (for torch 2.7.x),
# but Colab has torch 2.10 + torchvision 0.25. This breaks pip's resolver.
# Workaround: install both with --no-deps, then install dependencies separately.

# Step 1: Install sae-lens and transformer-lens without the torchvision constraint
!pip install --no-deps transformer-lens sae-lens

# Step 2: Install all actual dependencies (minus torchvision, not needed for text SAEs)
!pip install \
    "transformers>=5.4.0,<6.0.0" \
    "datasets>=3.1.0" \
    "einops>=0.6.0" \
    "jaxtyping>=0.2.11" \
    "fancy-einsum>=0.0.3" \
    "beartype>=0.14.1" \
    "accelerate>=0.23.0" \
    "rich>=12.6.0" \
    "sentencepiece" \
    "typeguard>=4.2,<5" \
    "wandb>=0.13.5" \
    "transformers-stream-generator>=0.0.5,<0.1" \
    "better-abc>=0.0.3" \
    "protobuf>=3.20.0" \
    "huggingface-hub>=0.23.2" \
    "safetensors>=0.4.2,<1.0.0" \
    "plotly>=5.19.0" \
    "nltk>=3.8.1" \
    "python-dotenv>=1.0.1" \
    "pyyaml>=6.0.1" \
    "simple-parsing>=0.1.8" \
    "tenacity>=9.0.0" \
    "babe>=0.0.7"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.2 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 36.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.3/276.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.9/236.9 kB 28.4 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=a5810aecf38dec850892768d3a6e5ef008538ba2659115e59c392b4534df6144
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b16

In [3]:
from huggingface_hub import login
import os
from __future__ import annotations

from tqdm import tqdm
import pandas as pd
import argparse
import re
import sys
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Iterable
from torch.utils.data import DataLoader

import requests
import torch, gc
from sae_lens import SAE
from transformer_lens import HookedTransformer

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive"
REPO_DIR = "/content/SAE_Surrogate"

gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))
print(torch.cuda.mem_get_info())
print(torch.cuda.is_bf16_supported())

Mounted at /content/drive
True
0
NVIDIA A100-SXM4-40GB
(41957130240, 42405855232)
True


In [4]:
with open(f"{DRIVE_DIR}/tokens/hftoken.txt") as f:
    token = f.readline().strip()

login(token=token)

In [5]:
with open(f"{DRIVE_DIR}/tokens/ghtoken.txt") as f:
    email = f.readline().strip()
    username = f.readline().strip()
    token = f.readline().strip()

!git config --global user.email "{email}"
!git config --global user.name "{username}"

#!git clone https://github.com/ntjohn04/SAE_Surrogate
if not os.path.exists(REPO_DIR):
    !git clone https://{token}@github.com/{username}/SAE_Surrogate
else:
    !git -C {REPO_DIR} pull

sys.path.append(f"{REPO_DIR}")

import util_neuronpedia
import util_nanogpt
import util_GSM8k

Cloning into 'SAE_Surrogate'...
remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 81 (delta 46), reused 58 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (81/81), 462.45 KiB | 5.08 MiB/s, done.
Resolving deltas: 100% (46/46), done.


# Load Models

In [6]:
DEFAULT_RELEASE = "gemma-scope-9b-it-res"
#DEFAULT_SAE_ID = "layer_20/width_16k/average_l0_71"
DEFAULT_SAE_ID = "layer_31/width_16k/average_l0_76"
DEFAULT_MODEL = "gemma-2-9b-it"
NEURONPEDIA_FEATURE_API = "https://www.neuronpedia.org/api/feature"

DEVICE = 'cuda'
DTYPE = torch.bfloat16


In [7]:
print("----loading SAE----")
sae = SAE.from_pretrained(
    release=DEFAULT_RELEASE,
    sae_id=DEFAULT_SAE_ID,
    device=DEVICE,
    dtype=DTYPE
)

print("----loading feature catalog----")
if not os.path.exists(f"{DRIVE_DIR}/feature_catalog_{DEFAULT_RELEASE}_{DEFAULT_SAE_ID.replace('/', '_')}.csv"):
    print("\tbuilding feature catalog...")
    feature_catalog = util_neuronpedia.build_feature_catalog()
    feature_catalog.to_csv(f"{DRIVE_DIR}/feature_catalog_{DEFAULT_RELEASE}_{DEFAULT_SAE_ID.replace('/', '_')}.csv", index=False)
else:
    feature_catalog = pd.read_csv(f"{DRIVE_DIR}/feature_catalog_{DEFAULT_RELEASE}_{DEFAULT_SAE_ID.replace('/', '_')}.csv")

print("----loading hooks----")
metadata = getattr(sae.cfg, "metadata", None)
hook_name = getattr(metadata, "hook_name", None)
match = re.search(r"blocks\.(\d+)\.", hook_name)
hook_layer = int(match.group(1)) if match else None

model_name = getattr(metadata, "model_name", None)
prepend_bos = getattr(metadata, "prepend_bos", None)

print(f"\tHook name:\t{hook_name}")
print(f"\tHook layer:\t{hook_layer}")
print(f"\tModel name:\t{model_name}")
print(f"\tPrepend BOS:\t{prepend_bos}")

print("----loading model----")
model = HookedTransformer.from_pretrained_no_processing(model_name,
                                          device=DEVICE, 
                                          dtype=DTYPE, 
                                          default_prepend_bos=prepend_bos, 
                                          #first_n_layers=hook_layer+1
                                          )

----loading SAE----


layer_31/width_16k/average_l0_76/params.(…):   0%|          | 0.00/470M [00:00<?, ?B/s]

----loading feature catalog----
----loading hooks----
	Hook name:	blocks.31.hook_resid_post
	Hook layer:	31
	Model name:	gemma-2-9b-it
	Prepend BOS:	True
----loading model----


config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model gemma-2-9b-it into HookedTransformer


# Functions

In [8]:
def safe_save(obj, path):
    tmp = str(path) + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

In [9]:
def get_feature_acts(sae, model, hook_name: str, token_ids: torch.Tensor) -> torch.Tensor:
    """Forward a padded token batch through model + SAE.

    Parameters
    ----------
    token_ids : LongTensor [batch, seq_len]  (on model device)

    Returns
    -------
    feature_acts : Tensor [batch, seq_len, num_features]
    """
    with torch.inference_mode():
        _, cache = model.run_with_cache(token_ids, names_filter=[hook_name])
        feature_acts = sae.encode(cache[hook_name])
    return feature_acts

# Data - GSM8K

In [10]:
train_ds, test_ds = util_GSM8k.load_gsm8k()
util_GSM8k.print_rows(train_ds, n=5, label="train")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

--- train (7473 rows) ---
[0] {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}
[1] {'question': 'Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?', 'answer': 'Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.\nWorking 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.\n#### 10'}
[2] {'question': 'Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?', 'answer': "In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.\nBetty's grandparents gave her 15 * 2 

# Generate Predictions and Activations

In [132]:
#train_ds = train_ds.select(range(3))
#test_ds  = test_ds.select(range(3))

In [11]:
records_path = f"{DRIVE_DIR}/records/{DEFAULT_RELEASE}_{DEFAULT_SAE_ID.replace('/', '_')}_gsm8k"

train_records_path = f"{records_path}_train_records.pt"
test_records_path  = f"{records_path}_test_records.pt"

train_gen_records_path = f"{records_path}_train_gen_records.pt"
test_gen_records_path  = f"{records_path}_test_gen_records.pt"

train_records = util_GSM8k.build_or_load(train_ds, train_records_path, model, 512, hook_name, sae, get_feature_acts, force=False)
test_records  = util_GSM8k.build_or_load(test_ds,  test_records_path, model, 512, hook_name, sae, get_feature_acts, force=False)

train_loader = DataLoader(train_records, batch_size=16, shuffle=False,  drop_last=False,  collate_fn=util_GSM8k.collate_gen)
test_loader  = DataLoader(test_records,  batch_size=16, shuffle=False, drop_last=False, collate_fn=util_GSM8k.collate_gen)

/usr/local/lib/python3.12/dist-packages/torch/_utils.py:358: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  result = torch.sparse_coo_tensor(


In [12]:
import os, glob

@torch.no_grad()
def generate_preds_sharded(loader, model, sae, hook_name, get_feature_acts,
                           preds_dir, flush_every=5, max_new_tokens=256, eot_id=107):
    os.makedirs(preds_dir, exist_ok=True)

    # --- resume: scan all existing shards for done ids ---
    done_ids = set()
    shard_paths = sorted(glob.glob(os.path.join(preds_dir, "shard_*.pt")))
    for sp in shard_paths:
        try:
            shard = torch.load(sp, weights_only=False)
            done_ids.update(p["id"] for p in shard)
        except Exception as e:
            print(f"WARNING: skipping unreadable shard {sp}: {e}")
    next_shard_idx = len(shard_paths)
    print(f"resuming: {len(done_ids)} already done across {len(shard_paths)} shards")

    buffer = []            # only UNFLUSHED records live here -> RAM stays flat
    batches_since_flush = 0

    def flush():
        nonlocal buffer, next_shard_idx, batches_since_flush
        if not buffer:
            return
        path = os.path.join(preds_dir, f"shard_{next_shard_idx:05d}.pt")
        tmp = path + ".tmp"
        torch.save(buffer, tmp)
        os.replace(tmp, path)          # rename to a NEW path (never overwrites) -> safe even on Drive
        next_shard_idx += 1
        buffer = []                    # evict from RAM
        batches_since_flush = 0

    for batch in tqdm(loader):
        batch_ids = batch["ids"].tolist()
        if all(_id in done_ids for _id in batch_ids):
            continue

        Pmax = batch["input_ids"].shape[1]
        gen = model.generate(batch["input_ids"].to(model.cfg.device),
                             attention_mask=batch["attention_mask"].to(model.cfg.device),
                             max_new_tokens=max_new_tokens, do_sample=False,
                             prepend_bos=False, stop_at_eos=True,
                             eos_token_id=eot_id, verbose=False)
        for j, _id in enumerate(batch_ids):
            if _id in done_ids:
                continue
            prompt = batch["input_ids"][j][batch["attention_mask"][j].bool()]
            ans = gen[j, Pmax:]
            cut = (ans == eot_id).nonzero()
            if len(cut): ans = ans[:cut[0, 0]]
            full = torch.cat([prompt.to(ans.device), ans]).cpu()
            fa = get_feature_acts(sae, model, hook_name,
                                  full[None].to(model.cfg.device))[0].half().cpu()
            buffer.append({"id": _id, "pred_input_ids": full,
                           "prompt_len": len(prompt),
                           "feature_acts_withpred": fa.to_sparse()})
            done_ids.add(_id)

        batches_since_flush += 1
        if batches_since_flush >= flush_every:
            flush()

    flush()  # final
    print(f"done: {len(done_ids)} total records in {next_shard_idx} shards")

def load_all_preds(preds_dir):
    preds = []
    for sp in sorted(glob.glob(os.path.join(preds_dir, "shard_*.pt"))):
        preds.extend(torch.load(sp, weights_only=False))
    # dedupe (in case a shard boundary ever double-wrote) and sort
    seen, out = set(), []
    for p in sorted(preds, key=lambda x: x["id"]):
        if p["id"] not in seen:
            seen.add(p["id"]); out.append(p)
    return out

In [14]:
def finalize_records(records, loader, model, sae, hook_name, get_feature_acts, preds_dir):
    generate_preds_sharded(loader, model, sae, hook_name, get_feature_acts, preds_dir)
    preds = load_all_preds(preds_dir)
    records.sort(key=lambda r: r["id"])
    assert len(records) == len(preds), f"{len(records)} != {len(preds)}"
    for r, p in zip(records, preds):
        assert r["id"] == p["id"]
        r.update({k: p[k] for k in ("pred_input_ids", "feature_acts_withpred")})
    n_maxed = sum(1 for p in preds if len(p["pred_input_ids"]) - p["prompt_len"] >= 256)
    print(f"answers that hit max_new_tokens (likely degenerate): {n_maxed}")
    return records

if not os.path.exists(train_gen_records_path):
    train_gen_records = finalize_records(train_records, train_loader, model, sae, hook_name, get_feature_acts, f"{records_path}_train_preds_shards")
    test_gen_records  = finalize_records(test_records,  test_loader,  model, sae, hook_name, get_feature_acts, f"{records_path}_test_preds_shards")

    safe_save(train_gen_records, train_gen_records_path)
    safe_save(test_gen_records, test_gen_records_path)
else:
    train_gen_records = torch.load(train_gen_records_path, weights_only=False)
    test_gen_records  = torch.load(test_gen_records_path,  weights_only=False)

resuming: 7473 already done across 63 shards


100%|██████████| 468/468 [00:00<00:00, 1259.59it/s]


done: 7473 total records in 63 shards
answers that hit max_new_tokens (likely degenerate): 377
resuming: 1319 already done across 17 shards


100%|██████████| 83/83 [00:00<00:00, 1278.68it/s]

done: 1319 total records in 17 shards


answers that hit max_new_tokens (likely degenerate): 87


# Records Exploration

In [23]:
print(len(train_gen_records), "train records with predictions")

for record in train_gen_records[:5]:
    print(record.keys())
    print(f"\tPred Shape:\t{record['feature_acts_withpred'].shape}")
    print(f"\tTrue Shape:\t{record['feature_acts_withtrue'].shape}")
    print(f"\tPrompt Len:\t{record['prompt_len']}")
    print(f"\tpred_input_ids:\t{record['pred_input_ids'].shape}")
    print(f"\tinput_ids:\t{record['input_ids'].shape}")

7473 train records with predictions
dict_keys(['id', 'prompt', 'true_answer', 'input_ids', 'prompt_len', 'feature_acts_withtrue', 'pred_input_ids', 'feature_acts_withpred'])
	Pred Shape:	torch.Size([141, 16384])
	True Shape:	torch.Size([102, 16384])
	Prompt Len:	45
	pred_input_ids:	torch.Size([141])
	input_ids:	torch.Size([102])
dict_keys(['id', 'prompt', 'true_answer', 'input_ids', 'prompt_len', 'feature_acts_withtrue', 'pred_input_ids', 'feature_acts_withpred'])
	Pred Shape:	torch.Size([151, 16384])
	True Shape:	torch.Size([106, 16384])
	Prompt Len:	40
	pred_input_ids:	torch.Size([151])
	input_ids:	torch.Size([106])
dict_keys(['id', 'prompt', 'true_answer', 'input_ids', 'prompt_len', 'feature_acts_withtrue', 'pred_input_ids', 'feature_acts_withpred'])
	Pred Shape:	torch.Size([191, 16384])
	True Shape:	torch.Size([172, 16384])
	Prompt Len:	69
	pred_input_ids:	torch.Size([191])
	input_ids:	torch.Size([172])
dict_keys(['id', 'prompt', 'true_answer', 'input_ids', 'prompt_len', 'feature_a

In [24]:
def sanity_check(records, name="records"):
    print(f"=== {name}: {len(records)} records ===")
    ids_seen = set()
    issues = 0

    for r in records:
        rid = r["id"]
        pl  = r["prompt_len"]
        ids_true = r["input_ids"]              # prompt + gold
        ids_pred = r["pred_input_ids"]         # prompt + generated
        fa_true  = r["feature_acts_withtrue"]
        fa_pred  = r["feature_acts_withpred"]

        def fail(msg):
            nonlocal issues; issues += 1
            print(f"  [id {rid}] {msg}")

        # 1. unique ids
        if rid in ids_seen: fail("duplicate id")
        ids_seen.add(rid)

        # 2. shapes: acts length must equal token length (acts are per-token)
        if fa_true.shape[0] != len(ids_true):
            fail(f"true acts len {fa_true.shape[0]} != input_ids len {len(ids_true)}")
        if fa_pred.shape[0] != len(ids_pred):
            fail(f"pred acts len {fa_pred.shape[0]} != pred_input_ids len {len(ids_pred)}")

        # 3. feature dim consistent
        if fa_true.shape[1] != fa_pred.shape[1]:
            fail(f"feature dim mismatch true={fa_true.shape[1]} pred={fa_pred.shape[1]}")

        # 4. prompt_len sane: positive, < both sequence lengths (answer is non-empty)
        if not (0 < pl <= len(ids_true)): fail(f"prompt_len {pl} out of range for true seq {len(ids_true)}")
        if not (0 < pl <= len(ids_pred)): fail(f"prompt_len {pl} out of range for pred seq {len(ids_pred)}")

        # 5. THE KEY ALIGNMENT INVARIANT: the prompt tokens must be IDENTICAL
        #    between the true sequence and the pred sequence (same prompt fed to both)
        pt_true = ids_true[:pl].cpu()
        pt_pred = ids_pred[:pl].cpu()
        if not torch.equal(pt_true, pt_pred):
            fail(f"prompt tokens differ between true and pred (first {pl} tokens)")

        # 6. prompt-span acts near-identical (noise floor ~0.999 cos, established earlier)
        import torch.nn.functional as F
        cs = F.cosine_similarity(fa_true.to_dense()[:pl].float(),
                                 fa_pred.to_dense()[:pl].float(), dim=-1).mean().item()
        if cs < 0.99:
            fail(f"prompt-span true/pred cosine {cs:.4f} < 0.99 (expected ~0.999 noise floor)")

        # 7. answer spans are non-empty (model actually generated something)
        if len(ids_pred) - pl <= 0: fail("empty predicted answer")
        if len(ids_true) - pl <= 0: fail("empty true answer")

        # 8. sparse acts are actually sparse (sanity on storage format)
        if fa_true.is_sparse:
            dens = fa_true._nnz() / fa_true.to_dense().numel()
            if dens > 0.1: fail(f"true acts density {dens:.3f} suspiciously high")

    # 9. ids form a contiguous-ish complete set
    print(f"  unique ids: {len(ids_seen)}  range: [{min(ids_seen)}, {max(ids_seen)}]")
    print(f"  issues found: {issues}")
    return issues == 0

ok = sanity_check(train_gen_records, "train")

=== train: 7473 records ===
  [id 2937] prompt-span true/pred cosine 0.9880 < 0.99 (expected ~0.999 noise floor)
  unique ids: 7473  range: [0, 7472]
  issues found: 1


# Surrogate Dataloaders

In [25]:
from torch.utils.data import Dataset, DataLoader

class SurrogateDataset(Dataset):
    """
    Serves per-example feature-vector sequences for next-feature-vector prediction.
    Each item: the dense [T, F] feature_acts_withpred sequence + prompt_len + id.
    Densification happens here (stored sparse), so the batch the model sees is dense.
    """
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        r = self.records[i]
        fa = r["feature_acts_withpred"]
        seq = fa.to_dense().float() if fa.is_sparse else fa.float()   # [T, F]
        return {
            "features": seq,                    # [T, F]
            "prompt_len": r["prompt_len"],      # answer starts here
            "id": r["id"],
        }


def surrogate_collate(batch):
    """
    Pads to [B, Tmax, F]. Builds shifted input/target for next-step prediction
    and a loss mask that (optionally) scores only answer-span transitions.

    Returns:
      inputs   : [B, Tmax-1, F]  feature vector at position t
      targets  : [B, Tmax-1, F]  feature vector at position t+1  (the thing to predict)
      loss_mask: [B, Tmax-1]     True where the transition should count toward loss
      prompt_lens, ids, lengths
    """
    B = len(batch)
    F = batch[0]["features"].shape[1]
    lengths = torch.tensor([b["features"].shape[0] for b in batch])
    Tmax = int(lengths.max())

    padded = torch.zeros(B, Tmax, F, dtype=torch.float32)
    for i, b in enumerate(batch):
        T = b["features"].shape[0]
        padded[i, :T] = b["features"]          # LEFT-align (pad on the right)

    # next-step pairs: predict position t+1 from position t
    inputs  = padded[:, :-1, :]                # [B, Tmax-1, F]
    targets = padded[:, 1:,  :]                # [B, Tmax-1, F]

    prompt_lens = torch.tensor([b["prompt_len"] for b in batch])
    ids         = torch.tensor([b["id"] for b in batch])

    # loss mask over the Tmax-1 transitions.
    # transition j predicts feature-vector at position j+1 from position j.
    # valid if j+1 < length (target is a real token, not padding).
    pos = torch.arange(Tmax - 1).unsqueeze(0)                      # [1, Tmax-1]
    real = (pos + 1) < lengths.unsqueeze(1)                        # target position is real
    # answer-span only: the PREDICTED position (j+1) is in the answer, i.e. >= prompt_len
    answer = (pos + 1) >= prompt_lens.unsqueeze(1)
    loss_mask_answer = real & answer                              # score only answer transitions
    loss_mask_all    = real                                       # score every real transition

    return {
        "inputs": inputs,
        "targets": targets,
        "loss_mask": loss_mask_answer,        # default: answer-span only
        "loss_mask_all": loss_mask_all,       # available if you want full-sequence scoring
        "prompt_lens": prompt_lens,
        "lengths": lengths,
        "ids": ids,
    }

surrogate_train_loader = DataLoader(
    SurrogateDataset(train_gen_records),
    batch_size=16, shuffle=True,  drop_last=True,
    collate_fn=surrogate_collate,
)
surrogate_test_loader = DataLoader(
    SurrogateDataset(test_gen_records),
    batch_size=16, shuffle=False, drop_last=False,
    collate_fn=surrogate_collate,
)

# Surrogate Training

In [ ]:
def masked_mse(predictions, targets, mask):
    """MSE over full 16384-dim vector, averaged over valid transitions only.
       Zero-dominated (mostly measures predicting the zeros) — the 'raw' number."""
    per_pos = ((predictions - targets) ** 2).mean(dim=-1)          # [B, T-1]
    return (per_pos * mask).sum() / mask.sum().clamp(min=1)

def masked_mse_active(predictions, targets, mask):
    """MSE restricted to features active in the target — the 'meaningful' number.
       Measures whether the model predicts the actual firings, not the zeros."""
    active = (targets > 0).float()                                 # [B, T-1, F]
    sq = ((predictions - targets) ** 2) * active
    per_pos = sq.sum(-1) / active.sum(-1).clamp(min=1)             # [B, T-1]
    return (per_pos * mask).sum() / mask.sum().clamp(min=1)

@torch.no_grad()
def masked_support_metrics(predictions, targets, mask, threshold=1e-3):
    """
    Binarize active features (pred > threshold, target > 0), compute confusion
    counts over valid transitions only, return precision/recall/F1/specificity.
    'Positive' = feature is active. Negatives dominate (~99.5%), so specificity
    will be near-perfect and uninformative; precision/recall/F1 are the signal.
    """
    pred_act = (predictions > threshold)                  # [B, T-1, F] bool
    targ_act = (targets > 0)
    m = mask.bool().unsqueeze(-1)                          # [B, T-1, 1]

    tp = (pred_act & targ_act & m).sum().float()
    fp = (pred_act & ~targ_act & m).sum().float()
    fn = (~pred_act & targ_act & m).sum().float()
    tn = (~pred_act & ~targ_act & m).sum().float()

    precision   = tp / (tp + fp).clamp(min=1)
    recall      = tp / (tp + fn).clamp(min=1)             # = sensitivity
    specificity = tn / (tn + fp).clamp(min=1)
    f1          = 2 * precision * recall / (precision + recall).clamp(min=1e-9)
    return {"precision": precision.item(), "recall": recall.item(),
            "f1": f1.item(), "specificity": specificity.item()}

def masked_cosine(predictions, targets, mask):
    """Mean cosine similarity between predicted and true vectors over valid transitions.
       Robust to the zero-domination; captures whether the active pattern matches."""
    cos = torch.nn.functional.cosine_similarity(predictions, targets, dim=-1)  # [B, T-1]
    return (cos * mask).sum() / mask.sum().clamp(min=1)

@torch.no_grad()
def topk_f1(model, loader, k=55, device="cuda"):
    TP=FP=FN=0.
    for b in loader:
        inp,tgt,m = b["inputs"].to(device),b["targets"].to(device),b["loss_mask"].to(device)
        pred = model.predict(inp)
        topk = pred.topk(k, dim=-1).indices                    # [B,T-1,k]
        pred_act = torch.zeros_like(pred, dtype=torch.bool).scatter_(-1, topk, True)
        ta = tgt>0; mm = m.bool().unsqueeze(-1)
        TP += (pred_act&ta&mm).sum().item()
        FP += (pred_act&~ta&mm).sum().item()
        FN += (~pred_act&ta&mm).sum().item()
    p=TP/max(TP+FP,1); r=TP/max(TP+FN,1)
    return {"topk_f1":2*p*r/max(p+r,1e-9),"precision":p,"recall":r}

@torch.no_grad()
def compute_train_mean(loader, F, device="cuda"):
    """Answer-span mean target vector, matching what evaluate() scores."""
    s = torch.zeros(F, device=device); n = 0.0
    for b in tqdm(loader):
        tgt = b["targets"].to(device); m = b["loss_mask"].to(device)
        s += tgt[m].sum(0); n += m.sum().item()
    return (s / max(n, 1)).cpu()

@torch.no_grad()
def compute_firing_stats(loader, F, device="cuda"):
    counts = torch.zeros(F, device=device)          # times each feature active
    sums   = torch.zeros(F, device=device)          # sum of values when active
    n = 0.0
    for b in loader:
        tgt, m = b["targets"].to(device), b["loss_mask"].to(device)
        t = tgt[m]                                   # [valid_positions, F]
        act = t > 0
        counts += act.sum(0); sums += (t * act).sum(0); n += t.shape[0]
    rates = counts / max(n, 1)                       # firing frequency per feature
    mean_active = sums / counts.clamp(min=1)         # mean value WHEN active (not diluted by zeros)
    avg_setsize = int(round((counts.sum() / max(n,1)).item()))   # ~55, sets k
    return rates.cpu(), mean_active.cpu(), avg_setsize

@torch.no_grad()
def evaluate(model, loader, device="cuda", threshold=1e-3):
    tot = {"mse":0.,"mse_active":0.,"cosine":0.}; tot_n = 0.
    TP=FP=FN=TN=0.
    for b in tqdm(loader):
        inp,tgt,m = b["inputs"].to(device), b["targets"].to(device), b["loss_mask"].to(device)
        pred = model.predict(inp)
        n = m.sum().item()
        if n==0: continue
        tot["mse"]        += masked_mse(pred,tgt,m).item()*n
        tot["mse_active"] += masked_mse_active(pred,tgt,m).item()*n
        tot["cosine"]     += masked_cosine(pred,tgt,m).item()*n
        tot_n += n
        pa,ta,mm = (pred>threshold), (tgt>0), m.bool().unsqueeze(-1)
        TP += (pa&ta&mm).sum().item(); FP += (pa&~ta&mm).sum().item()
        FN += (~pa&ta&mm).sum().item(); TN += (~pa&~ta&mm).sum().item()
    prec = TP/max(TP+FP,1); rec = TP/max(TP+FN,1)
    out = {k:v/max(tot_n,1) for k,v in tot.items()}
    out.update({"precision":prec, "recall":rec,
                "f1": 2*prec*rec/max(prec+rec,1e-9),
                "specificity": TN/max(TN+FP,1)})
    return out

def train_surrogate(model, loader, loss_fn, ckpt_dir, epochs=10, lr=1e-3,
                    save_every=2, clip=1.0, device="cuda", resume=True):
    os.makedirs(ckpt_dir, exist_ok=True)
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)

    # --- resume from latest checkpoint if present ---
    start_epoch = 0
    if resume:
        ckpts = sorted(glob.glob(os.path.join(ckpt_dir, "epoch_*.pt")))
        if ckpts:
            latest = ckpts[-1]
            state = torch.load(latest, weights_only=False)
            model.load_state_dict(state["model"])
            opt.load_state_dict(state["opt"])
            start_epoch = state["epoch"] + 1
            print(f"resumed from {latest} at epoch {start_epoch}")

    model.train()
    for ep in range(start_epoch, epochs):
        running, seen = 0.0, 0
        for b in tqdm(loader):
            inp = b["inputs"].to(device)
            tgt = b["targets"].to(device)
            m   = b["loss_mask"].to(device)
            pred = model.predict(inp)
            loss = loss_fn(pred, tgt, m)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
            running += loss.item(); seen += 1
        avg = running / max(seen, 1)
        print(f"epoch {ep}: train loss {avg:.4f}")

        # --- checkpoint every save_every epochs, and always on the last ---
        if (ep + 1) % save_every == 0 or ep == epochs - 1:
            path = os.path.join(ckpt_dir, f"epoch_{ep:03d}.pt")
            safe_save({"epoch": ep, "model": model.state_dict(),
                       "opt": opt.state_dict(), "loss": avg}, path)
            print(f"  saved {path}")
    return model

In [77]:
import torch.nn as nn

class LSTMSurrogate(nn.Module):
    """Nonlinear + recurrent history: compresses past into hidden state."""
    def __init__(self, F=16384, d_model=1024, hidden=1024, n_layers=2, p_drop=0.0):
        super().__init__()
        self.in_proj  = nn.Linear(F, d_model)        # 16384 -> d_model (like the transformer's embed)
        self.lstm     = nn.LSTM(d_model, hidden, num_layers=n_layers,
                                batch_first=True, dropout=p_drop)
        self.out_proj = nn.Linear(hidden, F)         # hidden -> 16384 (unembed)
    def predict(self, inputs):                       # [B, T-1, F] -> [B, T-1, F]
        x = self.in_proj(inputs)                     # project down
        h, _ = self.lstm(x)                          # causal by construction (processes left-to-right)
        return self.out_proj(h)  

class LinearSurrogate(nn.Module):
    """First-order linear: next = W·current + b. Memoryless, no nonlinearity."""
    def __init__(self, F=16384):
        super().__init__()
        self.lin = nn.Linear(F, F)
    def predict(self, inputs):               # [B, T-1, F] -> [B, T-1, F]
        return self.lin(inputs)              # applied per-position, no history

class MLPSurrogate(nn.Module):
    """Nonlinear memoryless: next = MLP(current). Still no history."""
    def __init__(self, F=16384, hidden=2048, n_layers=2, p_drop=0.0):
        super().__init__()
        dims = [F] + [hidden]*n_layers + [F]
        layers = []
        for i in range(len(dims)-1):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            if i < len(dims)-2:                        # nonlinearity between hidden layers, not on output
                layers += [nn.GELU(), nn.Dropout(p_drop)]
        self.net = nn.Sequential(*layers)
    def predict(self, inputs):
        return self.net(inputs)

class PersistenceBaseline:
    """Predict next vector = current vector. No parameters."""
    def predict(self, inputs):                # inputs: [B, T-1, F]
        return inputs

class MeanBaseline:
    """Predict the training-set mean vector everywhere. No training."""
    def __init__(self, mean_vec):             # mean_vec: [F]
        self.mean = mean_vec
    def predict(self, inputs):
        B, T, F = inputs.shape
        return self.mean.to(inputs.device).view(1, 1, F).expand(B, T, F)
    
class BaseRateBaseline:
    """Predict the k most-frequently-active features as 'on' (with their mean-active value),
       everything else off. k chosen to match the true average active-set size."""
    def __init__(self, firing_rates, mean_active_vals, k):
        topk = torch.topk(firing_rates, k).indices
        self.vec = torch.zeros_like(mean_active_vals)
        self.vec[topk] = mean_active_vals[topk]   # predict their typical magnitude
    def predict(self, inputs):
        B, T, F = inputs.shape
        return self.vec.to(inputs.device).view(1,1,F).expand(B,T,F)

In [ ]:
F = 16384

mean_vec = compute_train_mean(surrogate_train_loader, F)
mean_model = MeanBaseline(mean_vec)

rates, mean_active, k = compute_firing_stats(surrogate_train_loader, 16384)
baserate = BaseRateBaseline(rates, mean_active, k)

persistence = PersistenceBaseline()

lin = LinearSurrogate().to("cuda")
train_surrogate(lin, surrogate_train_loader, masked_mse,
                ckpt_dir=f"{records_path}_ckpt_linear", epochs=10)

mlp = MLPSurrogate().to("cuda")
train_surrogate(mlp, surrogate_train_loader, masked_mse,
                ckpt_dir=f"{records_path}_ckpt_mlp", epochs=10)

lstm = LSTMSurrogate().to("cuda")
train_surrogate(lstm, surrogate_train_loader, masked_mse,
                ckpt_dir=f"{records_path}_ckpt_lstm", epochs=10)

In [58]:
print("mean:", evaluate(mean_model, surrogate_test_loader))
print("persistence:", evaluate(persistence, surrogate_test_loader))
print("base-rate:", evaluate(baserate, surrogate_test_loader))
print("linear:", evaluate(lin, surrogate_test_loader))
print("mlp:", evaluate(mlp, surrogate_test_loader))
print("lstm:", evaluate(lstm, surrogate_test_loader))

print("mean top-k:", topk_f1(mean_model, surrogate_test_loader))
print("persistence top-k:",    topk_f1(persistence, surrogate_test_loader))
print("base-rate top-k:",     topk_f1(baserate, surrogate_test_loader))
print("linear top-k:", topk_f1(lin, surrogate_test_loader))
print("mlp top-k:",    topk_f1(mlp, surrogate_test_loader))
print("lstm top-k:", topk_f1(lstm, surrogate_test_loader))

persistence: {'mse': 3.9414584435976376, 'mse_active': 438.90173379862256, 'cosine': 0.46474654173277413, 'precision': 0.3682316165702686, 'recall': 0.36892813035604266, 'f1': 0.3685795444086625, 'specificity': 0.9967149912007686}


100%|██████████| 467/467 [01:28<00:00,  5.28it/s]


mean: {'mse': 2.8988533113955826, 'mse_active': 531.2727287788367, 'cosine': 0.46836850622868814, 'precision': 0.006863210339249418, 'recall': 0.9986660809265103, 'f1': 0.0136327314014784, 'specificity': 0.25000215865262376}


# Extra